In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle

In [3]:
df = pd.read_csv('/content/train.csv')

In [4]:
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop(columns=['Cabin'], inplace=True)  # Drop Cabin due to too many missing values

/tmp/ipython-input-2093214286.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipython-input-2093214286.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

In [5]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [6]:
label_encoders = {}
for col in ['Sex', 'Embarked']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [7]:
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']]
y = df['Survived']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [9]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

In [10]:
cv_results = {}
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='accuracy')
    cv_results[name] = scores.mean()

In [11]:
best_model_name = max(cv_results, key=cv_results.get)
best_model = models[best_model_name]

In [12]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

In [13]:
final_accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

In [38]:
import os
os.makedirs("/content/model", exist_ok=True)  # Colab path
with open("/content/model/passenger_survival_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

In [39]:
from google.colab import files
files.download("/content/model/passenger_survival_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
cv_results, best_model_name, final_accuracy, report

({'Logistic Regression': np.float64(0.8005811090318133),
  'Random Forest': np.float64(0.8005712597261894)},
 'Logistic Regression',
 0.8100558659217877,
 '              precision    recall  f1-score   support\n\n         0.0       0.82      0.89      0.85       110\n         1.0       0.80      0.68      0.73        69\n\n    accuracy                           0.81       179\n   macro avg       0.81      0.79      0.79       179\nweighted avg       0.81      0.81      0.81       179\n')